# Install TabPFN Extenstions

In [ ]:
import importlib.util
import sys

def _missing(mod):
    return importlib.util.find_spec(mod) is None

to_install = []
# `shap` is only needed for the plotting API in the SHAP section — shapiq
# (installed via tabpfn-extensions[all]) does the actual computation.
if _missing("shap"):
    to_install.append("shap")

if to_install:
    print("Installing:", to_install)
    # `-q -q` = silence "Collecting/Downloading" progress lines.
    # `--no-warn-conflicts` = silence pre-existing Kaggle image conflicts
    # (google-colab, moviepy, bigframes, ...) that this notebook doesn't touch.
    get_ipython().run_line_magic("pip", "install -q -q --no-warn-conflicts " + " ".join(to_install))
    print("Done. If imports fail, restart the kernel and re-run.")
else:
    print("shap already available.")

In [ ]:
print("Installing tabpfn-extensions[interpretability]... (may take ~1 minute)")
# Only the `interpretability` extra is needed for this notebook.
# `[all]` also pulls autogluon / hyperopt / llvmlite via `post_hoc_ensembles`
# + `hpo`, which are ~150 MB of unrelated deps and downgrade `pyarrow`,
# triggering the pip resolver conflict noise we don't want.
# `-q -q --no-warn-conflicts` silences "Collecting/Downloading" progress
# lines and the pre-existing Kaggle image conflict warnings (google-colab,
# moviepy, bigframes, ...) that this notebook doesn't touch.
%pip install -q -q --no-warn-conflicts "tabpfn-extensions[interpretability] @ git+https://github.com/PriorLabs/tabpfn-extensions.git"
%pip install -q -q --no-warn-conflicts "tabpfn-client"
%pip install -q -q --no-warn-conflicts "tabpfn"
print("Done. If imports fail, restart the kernel and re-run.")

### Load env

In [ ]:
import os
import sys
import warnings
from contextlib import contextmanager
from io import StringIO

# Auto-detect Kaggle.
is_kaggle_env = os.path.isdir("/kaggle/working")

DEVICE = "cpu"
DEVICE_NAME = "CPU"
_gpu_note = ""

try:
    import torch

    # torch emits noisy "compute capability / sm_XX not supported" UserWarnings
    # on first CUDA touch for GPUs older than the ones the installed wheel was
    # built for (e.g. Kaggle's Tesla P100 = sm_60 with a torch built for
    # sm_70+). We handle the fallback explicitly below, so silence them.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"(?s).*(compute capability|CUDA capability|sm_\d+).*",
            category=UserWarning,
        )
        if torch.cuda.is_available():
            _major, _minor = torch.cuda.get_device_capability(0)
            _gpu_name = torch.cuda.get_device_name(0)
            _supported = torch.cuda.get_arch_list()  # e.g. ['sm_70', ..., 'sm_120']
            _min_major = min(
                (int(a.removeprefix("sm_")[:-1]) for a in _supported if a.startswith("sm_")),
                default=7,
            )
            if _major >= _min_major:
                DEVICE = "cuda"
                DEVICE_NAME = _gpu_name
            else:
                # Kernel launch would crash -- fall back to CPU with a hint.
                _gpu_note = (
                    f" | note: GPU {_gpu_name} (sm_{_major}{_minor}) is below "
                    f"the installed torch build's minimum (sm_{_min_major}0) -- "
                    f"switch the Kaggle accelerator to T4 x2 for a working GPU"
                )
except Exception:
    pass


def _load_tabpfn_tokens():
    hf = os.environ.get("HF_TOKEN", "").strip()
    tabpfn = os.environ.get("TABPFN_TOKEN", "").strip()
    if tabpfn:
        return hf, tabpfn
    if is_kaggle_env:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        hf = secrets.get_secret("HF_TOKEN")
        tabpfn = secrets.get_secret("NEW_TABPFN_TOKEN") or secrets.get_secret("TABPFN_TOKEN")
        return hf, tabpfn
    return hf, tabpfn

_hf_token, _tabpfn_token = _load_tabpfn_tokens()
if _tabpfn_token:
    os.environ["TABPFN_TOKEN"] = _tabpfn_token
if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token

os.environ["USE_TABPFN_LOCAL"] = "false"

# Quota knobs (tabpfn-client free tier, as of 2026):
#   - 20 thinking fits
#   - 50M prediction cells / day
#   - 200M prediction cells / month
# Hybrid backend:
#   - Feature selection: local `tabpfn` on GPU/CPU (0 client API calls)
#   - PDP / SHAP / SHAP-IQ: tabpfn-client + thinking (~3 thinking fits)
# USE_TABPFN_LOCAL=false keeps tabpfn_extensions on the client if imported
# elsewhere; the FS cell imports `from tabpfn import TabPFNClassifier` directly.
FS_THINKING_MODE = False
INTERP_THINKING_MODE = True
INTERP_THINKING_EFFORT = "high"
INTERP_THINKING_METRIC = "average_precision"


def _silence_client_progress(clf):
    """Prefer the official switch when present (tabpfn-client show_progress)."""
    if hasattr(clf, "show_progress"):
        clf.show_progress = False
    return clf


@contextmanager
def _quiet_tabpfn_client():
    """Discard tabpfn-client Fitting/Predicting spinner writes to stdout.

    Does not silence our own print() trackers -- only wrap fit/predict/explain
    blocks that trigger the client's run_task spinner.
    """
    _real_stdout = sys.stdout
    try:
        sys.stdout = StringIO()
        yield
    finally:
        sys.stdout = _real_stdout


print(
    f"Runtime: {'Kaggle' if is_kaggle_env else 'local'} | "
    f"TabPFN device: {DEVICE} ({DEVICE_NAME}) | "
    f"Tokens: {bool(_tabpfn_token) and bool(_hf_token)}"
    f"{_gpu_note}"
)
print(
    "Quota (client): 20 thinking fits | 50M cells/day | 200M cells/month"
)
print(
    "This notebook intends ~3 client thinking fits; FS uses local tabpfn "
    "(0 client API calls)."
)
print(
    f"Thinking: FS={FS_THINKING_MODE} | "
    f"PDP/SHAP/SHAP-IQ={INTERP_THINKING_MODE} "
    f"(effort={INTERP_THINKING_EFFORT}, metric={INTERP_THINKING_METRIC})"
)



### Load data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_MODE = "raw"            # "raw" (spec default) or "processed"

TARGET_COL = "Stent thrombosis"
DROP_FEATURES = ["Time since stent implantation"]   # leakage control
ID_COLS = ["NO.", "Name"]
RANDOM_STATE = 42
TEST_SIZE = 0.3

KAGGLE_RAW_CSV = "/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv"
KAGGLE_PROCESSED_DIR = "/kaggle/input/datasets/amirmahdidaraei/preprocessed-data"
KAGGLE_RESULT_SUBDIR = "modeling_tabpfn"
RESULT_DIR = None  # resolved in the load cell

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "data" / "raw" / "VLST.csv").is_file():
            return d
    raise FileNotFoundError(
        "Could not locate data/raw/VLST.csv above the current working directory."
    )


def _discover_vlst_csv() -> Path:
    """Resolve VLST.csv on Kaggle (env override, then recursive search under /kaggle/input)."""
    env = os.environ.get("VLST_RAW_CSV")
    if env and Path(env).is_file():
        return Path(env)

    candidates = [
        Path(KAGGLE_RAW_CSV),
        Path("/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv"),
        Path("/kaggle/input/vlst-data/VLST.csv"),
        Path("/kaggle/input/VLST_data/VLST.csv"),
        Path("/kaggle/input/datasets/amirinho661/vlst-figshare-7409606/VLST.csv")
    ]
    for p in candidates:
        if p.is_file():
            return p

    base = Path("/kaggle/input")
    if base.is_dir():
        for p in base.rglob("VLST.csv"):
            if p.is_file():
                return p

    raise FileNotFoundError(
        "VLST.csv not found on Kaggle. Upload it as a dataset (see §0) or set "
        "os.environ['VLST_RAW_CSV'] = '/kaggle/input/<dataset>/VLST.csv'."
    )


def _resolve_paths():
    """Return (raw_path, processed_dir, result_dir, label) for local or Kaggle."""
    if is_kaggle_env:
        raw = _discover_vlst_csv()
        processed = Path(
            os.environ.get("VLST_PROCESSED_DIR", KAGGLE_PROCESSED_DIR)
        )
        result = Path(
            os.environ.get(
                "VLST_RESULT_DIR",
                str(Path("/kaggle/working") / KAGGLE_RESULT_SUBDIR),
            )
        )
        return raw, processed, result, f"Kaggle | raw={raw}"

    repo = _find_repo_root()
    return (
        repo / "data" / "raw" / "VLST.csv",
        repo / "data" / "processed",
        repo / "data" / "result" / "modeling_tabpfn",
        f"local | repo={repo}",
    )


RAW_PATH, PROCESSED_DIR, RESULT_DIR, _path_label = _resolve_paths()
RESULT_DIR = Path(RESULT_DIR)
PROCESSED_DIR = Path(PROCESSED_DIR)
RAW_PATH = Path(RAW_PATH)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(_path_label)
print("RAW_PATH:", RAW_PATH)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULT_DIR:", RESULT_DIR)


def load_raw():
    """Minimal, TabPFN-native handling: keep NaNs, code text columns, no scaling/one-hot."""
    df = pd.read_csv(RAW_PATH)
    df = df.drop(columns=[c for c in ID_COLS if c in df.columns])
    y = df[TARGET_COL].astype(int).to_numpy()
    drop = [TARGET_COL] + [c for c in DROP_FEATURES if c in df.columns]
    X_df = df.drop(columns=drop)
    for c in X_df.columns:
        if X_df[c].dtype == object:
            coerced = pd.to_numeric(X_df[c].astype(str).str.strip(), errors="coerce")
            if coerced.notna().mean() >= 0.5:        # genuinely numeric (e.g. "21.00 ")
                X_df[c] = coerced
            else:                                     # categorical text -> integer codes (NOT one-hot)
                codes = X_df[c].astype("category").cat.codes.astype(float)
                X_df[c] = codes.where(codes >= 0, np.nan)
    return X_df.to_numpy(dtype=float), y, list(X_df.columns)

X_all, y_all, feature_names = load_raw()
print("Loaded RAW VLST.csv")

print(f"X: {X_all.shape} | y: {y_all.shape} | Features: {len(feature_names)}")

### Feature selection — [1/4]

Local rankings (0 tabpfn-client calls):

1. **mutual_info_classif** — fast univariate screen → `interpretability_mutual_info_ranking.csv`
2. **TabPFN forward SFS** — model-native CV PR-AUC → `interpretability_forward_feature_selection.csv` (used by PDP)
3. **TabPFN backward SFS** (optional, `RUN_BACKWARD_FS=False` by default) → `interpretability_backward_feature_selection.csv`

In [ ]:
"""Feature selection for VLST interpretability — local methods.

1. mutual_info_classif — fast univariate ranking (seconds, 0 TabPFN calls).
2. Local TabPFN forward SFS — CV PR-AUC with the same model family as PDP/SHAP
   (`from tabpfn import TabPFNClassifier`, not tabpfn-client).
3. Local TabPFN backward SFS — optional via RUN_BACKWARD_FS (default False; ~2–3x cost).

TabPFN SFS on the *client* burns the 50M daily quota; local tabpfn avoids that.
Forward by default; n_jobs=1, n_estimators=1 keeps wall time manageable on a T4.

CAUTION — data leakage if you reuse these features downstream:
All steps rank on the full (X_all, y_all) pool for *interpretability only*.
Treat the CSVs as exploratory guidance, not a locked-in feature mask.
PDP reads `interpretability_forward_feature_selection.csv` (TabPFN forward SFS).
"""

from __future__ import annotations

import time

from sklearn.feature_selection import mutual_info_classif
from tabpfn import TabPFNClassifier
from tabpfn_extensions import interpretability

print("=" * 60)
print("[1/4] Feature selection (mutual_info + local TabPFN SFS)")
print("=" * 60)

SKIP_FS_IF_EXISTS = True
RUN_BACKWARD_FS = True  # True = also run local TabPFN backward SFS (~2–3x cost)

_mi_csv = RESULT_DIR / "interpretability_mutual_info_ranking.csv"
_fs_forward = RESULT_DIR / "interpretability_forward_feature_selection.csv"
_fs_backward = RESULT_DIR / "interpretability_backward_feature_selection.csv"

N_MI_RANK = min(15, X_all.shape[1])
N_FEATURES_TO_SELECT = min(10, X_all.shape[1])
FS_CV = 3
FS_N_JOBS = 1
FS_N_ESTIMATORS = 1


def _make_fs_clf():
    return TabPFNClassifier(
        device=DEVICE,
        n_estimators=FS_N_ESTIMATORS,
        balance_probabilities=True,
        ignore_pretraining_limits=True,
        random_state=RANDOM_STATE,
    )


_t0 = time.perf_counter()

# --- 1a. Mutual information (univariate) ---------------------------------
print("\n--- [1a] mutual_info_classif ---")
if SKIP_FS_IF_EXISTS and _mi_csv.is_file():
    print(f"Skipping MI ranking — already saved at {_mi_csv}")
    print("(Delete the CSV to re-run.)")
    _mi_ranked = pd.read_csv(_mi_csv)
else:
    print(
        f"Ranking {X_all.shape[1]} features with mutual_info_classif "
        f"(top {N_MI_RANK})..."
    )
    _col_med = np.nanmedian(X_all, axis=0)
    _X_mi = np.where(np.isnan(X_all), _col_med, X_all)
    _mi = mutual_info_classif(
        _X_mi,
        y_all,
        discrete_features=False,
        random_state=RANDOM_STATE,
    )
    _order = np.argsort(_mi)[::-1]
    _top = _order[:N_MI_RANK]
    _mi_ranked = pd.DataFrame(
        {
            "rank": range(1, len(_top) + 1),
            "feature": [feature_names[i] for i in _top],
            "mutual_info": [_mi[i] for i in _top],
        },
    )
    _mi_ranked.to_csv(_mi_csv, index=False)
    print(f"Top {N_MI_RANK}: {_mi_ranked['feature'].tolist()}")
    print(f"Saved: {_mi_csv}")

# --- 1b. Local TabPFN forward sequential FS -----------------------------
print("\n--- [1b] local TabPFN forward SFS ---")
if SKIP_FS_IF_EXISTS and _fs_forward.is_file():
    print(f"Skipping TabPFN forward SFS — already saved at {_fs_forward}")
    print("(Delete the CSV to re-run.)")
    _tabpfn_ranked = pd.read_csv(_fs_forward)
    print(f"Loaded {len(_tabpfn_ranked)} selected features from existing CSV.")
else:
    print(
        f"Forward SFS: select {N_FEATURES_TO_SELECT} of {X_all.shape[1]} features, "
        f"cv={FS_CV}, device={DEVICE} (local tabpfn; 0 client API calls)..."
    )

    result = interpretability.feature_selection.feature_selection(
        estimator=_make_fs_clf(),
        X=X_all,
        y=y_all,
        n_features_to_select=N_FEATURES_TO_SELECT,
        feature_names=list(feature_names),
        cv=FS_CV,
        scoring="average_precision",
        direction="forward",
        n_jobs=FS_N_JOBS,
        tol=0.0001,
        verbose=True,
    )

    print("\nProgrammatic summary (forward):")
    print(f"Selected features ({len(result.selected_names)}): {result.selected_names}")
    print(
        f"CV score before / after: "
        f"{result.baseline_score_mean:.4f} -> {result.selected_score_mean:.4f}"
    )

    _tabpfn_ranked = pd.DataFrame(
        {
            "rank": range(1, len(result.selected_names) + 1),
            "feature": result.selected_names,
        },
    )
    _tabpfn_ranked.to_csv(_fs_forward, index=False)
    print(f"Saved: {_fs_forward}")

# --- 1c. Local TabPFN backward sequential FS (optional) -----------------
if RUN_BACKWARD_FS:
    print("\n--- [1c] local TabPFN backward SFS ---")
    if SKIP_FS_IF_EXISTS and _fs_backward.is_file():
        print(f"Skipping TabPFN backward SFS — already saved at {_fs_backward}")
        print("(Delete the CSV to re-run.)")
    else:
        print(
            f"Backward SFS: select {N_FEATURES_TO_SELECT} of {X_all.shape[1]} features, "
            f"cv={FS_CV}, device={DEVICE} (much slower than forward)..."
        )

        result_bwd = interpretability.feature_selection.feature_selection(
            estimator=_make_fs_clf(),
            X=X_all,
            y=y_all,
            n_features_to_select=N_FEATURES_TO_SELECT,
            feature_names=list(feature_names),
            cv=FS_CV,
            scoring="average_precision",
            direction="backward",
            n_jobs=FS_N_JOBS,
            tol=0.0001,
            verbose=True,
        )

        print("\nProgrammatic summary (backward):")
        print(
            f"Selected features ({len(result_bwd.selected_names)}): "
            f"{result_bwd.selected_names}"
        )
        print(
            f"CV score before / after: "
            f"{result_bwd.baseline_score_mean:.4f} -> "
            f"{result_bwd.selected_score_mean:.4f}"
        )

        pd.DataFrame(
            {
                "rank": range(1, len(result_bwd.selected_names) + 1),
                "feature": result_bwd.selected_names,
            },
        ).to_csv(_fs_backward, index=False)
        print(f"Saved: {_fs_backward}")
else:
    print("\n--- [1c] local TabPFN backward SFS — skipped (RUN_BACKWARD_FS=False) ---")

_elapsed = time.perf_counter() - _t0
_artifacts = f"MI: {_mi_csv} | TabPFN forward: {_fs_forward}"
if RUN_BACKWARD_FS:
    _artifacts += f" | TabPFN backward: {_fs_backward}"
print(f"\n[1/4] done in {_elapsed:.1f}s | {_artifacts}")




### PDP (Partial Dependence Plot) Part

In [ ]:
"""Partial dependence plots for the VLST TabPFN classifier.

PDP issues many predicts (`grid_resolution * n_features` for 1D plots;
more for 2D interactions) against the same fitted model, so the KV cache
(`fit_mode='fit_with_cache'`) avoids redoing the encoder pass over
X_train on every grid point. Falls back to the default constructor for
backends/versions that don't support the cache -- that's tabpfn-client
(TypeError on the kwarg) and older local tabpfn (accepts the kwarg but
raises ValueError/NotImplementedError at fit time).
"""

from __future__ import annotations

import time
import warnings

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from tabpfn_client import TabPFNClassifier
from tabpfn_extensions.interpretability.pdp import partial_dependence_plots

print("=" * 60)
print("[2/4] PDP (Partial Dependence Plots)")
print("=" * 60)

SKIP_PDP_IF_EXISTS = True
pdp_out = RESULT_DIR / "interpretability_pdp.png"

_t0 = time.perf_counter()

if SKIP_PDP_IF_EXISTS and pdp_out.is_file():
    print(f"Skipping PDP -- already saved at {pdp_out}")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all,
        test_size=TEST_SIZE,
        stratify=y_all,           # VLST is imbalanced -- always stratify
        random_state=RANDOM_STATE,
    )

    def _make_pdp_clf(**extra):
        return TabPFNClassifier(
            balance_probabilities=True,
            thinking_mode=INTERP_THINKING_MODE,
            thinking_effort=INTERP_THINKING_EFFORT,
            thinking_metric=INTERP_THINKING_METRIC,
            random_state=42,
        )

    print("Fitting TabPFN for PDP (1 thinking fit)...")
    try:
        clf = _silence_client_progress(_make_pdp_clf(fit_mode="fit_with_cache"))
        with _quiet_tabpfn_client():
            clf.fit(X_train, y_train)
        if hasattr(clf, "executor_"):
            clf.executor_.keep_cache_on_device = True
    except (TypeError, ValueError, NotImplementedError):
        warnings.warn(
            "PDP would benefit substantially from the KV cache, but the "
            "current TabPFN install doesn't support fit_mode='fit_with_cache' "
            "(typical of older tabpfn versions or the tabpfn-client backend). "
            "Upgrade to the latest version of tabpfn (`pip install -U tabpfn`) "
            "for a substantial speedup on this example. Falling back to the "
            "default constructor.",
            UserWarning,
            stacklevel=2,
        )
        clf = _silence_client_progress(_make_pdp_clf())
        with _quiet_tabpfn_client():
            clf.fit(X_train, y_train)

    # 1D PD for top-3 from FS CSV + one pairwise interaction.
    _fs_csv = RESULT_DIR / "interpretability_forward_feature_selection.csv"
    if _fs_csv.is_file():
        _pd_names = pd.read_csv(_fs_csv)["feature"].tolist()[:3]
    else:
        _pd_names = list(feature_names[:3])
    _pd_idx = [feature_names.index(n) for n in _pd_names]
    _interact_partner = (
        feature_names[3]
        if _pd_names[0] != feature_names[3]
        else feature_names[min(4, len(feature_names) - 1)]
    )
    pd_features = _pd_idx + [(_pd_idx[0], feature_names.index(_interact_partner))]
    print(
        f"Computing PDP: features={_pd_names} + interaction "
        f"({_pd_names[0]}, {_interact_partner}), grid_resolution=30..."
    )

    with _quiet_tabpfn_client():
        disp = partial_dependence_plots(
            estimator=clf,
            X=X_test,
            features=pd_features,
            grid_resolution=30,
            kind="average",
            target_class=1,       # positive class = "Stent thrombosis"
            feature_names=list(feature_names),
        )
    disp.figure_.suptitle("Partial dependence -- VLST (P[Stent thrombosis])")

    plt.savefig(pdp_out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {pdp_out}")

_elapsed = time.perf_counter() - _t0
print(f"[2/4] done in {_elapsed:.1f}s | artifact: {pdp_out}")




### SHAP Part

In [ ]:
"""SHAP values for the VLST TabPFN classifier via shapiq, visualized with
the SHAP library's plotting API.

We use shapiq for the actual Shapley-value computation (faster and
extension-friendly for TabPFN) but the SHAP library's plotting ecosystem
is mature. This bridges the two: wrap shapiq output in a
`shap.Explanation` and call `shap.plots.*` / `shap.summary_plot`.

For k-SII (pairwise interactions), standard SHAP plots only show first-order
effects -- the k-SII block below uses shapiq's native `plot_network` /
`plot_upset` instead.

For a classifier, `get_tabpfn_imputation_explainer` defaults to
`class_index=1` -- the positive class. That's what we want on VLST (target
= "Stent thrombosis").

The `shap` package is not part of the `interpretability` extra (we depend
on shapiq for compute). It's installed by the top-of-notebook install
cell.

VLST has ~50 features, so 2**d exact enumeration is astronomical -- the
budget below samples coalitions instead. Increase it on GPU for tighter
estimates.

The TabPFN model is constructed with `fit_mode="fit_with_cache"` to
engage the KV cache, which speeds up Shapley-value computation by one to
two orders of magnitude; the shapiq wrapper warns if the cache isn't
enabled.
"""

from __future__ import annotations

import time
import warnings

import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split

from tabpfn_client import TabPFNClassifier
from tabpfn_extensions.interpretability import shapiq as tabpfn_shapiq

print("=" * 60)
print("[3/4] SHAP (SV + one-row k-SII)")
print("=" * 60)

SKIP_SHAP_IF_EXISTS = True
_shap_plots = [
    RESULT_DIR / "sv_interpretability_shap_summary.png",
    RESULT_DIR / "sv_interpretability_shap_scatter_f0.png",
    RESULT_DIR / "sv_interpretability_shap_bar.png",
    RESULT_DIR / "sv_interpretability_shap_beeswarm.png",
    RESULT_DIR / "sv_interpretability_shap_waterfall_row0.png",
    RESULT_DIR / "k_ssi_interpretability_network.png",
    RESULT_DIR / "k_ssi_interpretability_upset.png",
]

_t0 = time.perf_counter()

if SKIP_SHAP_IF_EXISTS and all(p.is_file() for p in _shap_plots):
    print(
        "Skipping SHAP -- all plot artifacts already exist under "
        f"{RESULT_DIR}"
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all,
        test_size=TEST_SIZE,
        stratify=y_all,
        random_state=RANDOM_STATE,
    )

    # Explain positives first so the beeswarm/waterfall highlight the
    # minority (informative) class before falling back to negatives.
    _pos_idx = np.where(y_test == 1)[0]
    _neg_idx = np.where(y_test == 0)[0]
    SHAP_N_EXPLAIN = min(15, X_test.shape[0])
    _order = np.concatenate([_pos_idx, _neg_idx])[:SHAP_N_EXPLAIN]
    X_explain = X_test[_order]

    def _make_shap_clf(**extra):
        return TabPFNClassifier(
            balance_probabilities=True,
            thinking_mode=INTERP_THINKING_MODE,
            thinking_effort=INTERP_THINKING_EFFORT,
            thinking_metric=INTERP_THINKING_METRIC,
            random_state=42,
        )

    print("Fitting TabPFN for SHAP (1 thinking fit)...")
    try:
        clf = _silence_client_progress(_make_shap_clf(fit_mode="fit_with_cache"))
        with _quiet_tabpfn_client():
            clf.fit(X_train, y_train)
    except (TypeError, ValueError, NotImplementedError):
        warnings.warn(
            "SHAP would benefit substantially from the KV cache, but the "
            "current TabPFN install doesn't support fit_mode='fit_with_cache' "
            "(typical of older tabpfn versions or the tabpfn-client backend). "
            "Falling back to the default constructor.",
            UserWarning, stacklevel=2,
        )
        clf = _silence_client_progress(_make_shap_clf())
        with _quiet_tabpfn_client():
            clf.fit(X_train, y_train)

    # `class_index` defaults to 1 for classifiers -- explains P(positive).
    explainer = tabpfn_shapiq.get_tabpfn_imputation_explainer(
        model=clf,
        data=X_train,
        index="SV",
        imputer="baseline",
        max_order=1,
    )

    SHAPIQ_BUDGET = 256

    # Per-row loop (mirrors shapiq_to_shap_explanation) so we can print
    # progress between rows without client spinner spam.
    print(
        f"Explaining {SHAP_N_EXPLAIN}/{SHAP_N_EXPLAIN} rows, "
        f"budget={SHAPIQ_BUDGET}, imputer=baseline..."
    )
    _ivs = []
    for _i in range(SHAP_N_EXPLAIN):
        with _quiet_tabpfn_client():
            _ivs.append(explainer.explain(x=X_explain[_i], budget=SHAPIQ_BUDGET))
        print(f"SHAP row {_i + 1}/{SHAP_N_EXPLAIN} done")

    _values = np.stack([iv.get_n_order_values(1) for iv in _ivs])
    _base_values = np.array([iv.baseline_value for iv in _ivs])
    explanation = shap.Explanation(
        values=_values,
        base_values=_base_values,
        data=X_explain,
        feature_names=list(feature_names),
    )

    # 1. Summary plot -- beeswarm of feature attributions across all explained rows.
    shap.summary_plot(explanation, show=False)
    plt.savefig(RESULT_DIR / "sv_interpretability_shap_summary.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 2. Scatter -- SHAP value of feature 0 vs. its raw value, colored by the
    #    feature shap picks as its strongest interaction partner.
    shap.plots.scatter(explanation[:, 0], show=False)
    plt.savefig(RESULT_DIR / "sv_interpretability_shap_scatter_f0.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 3. Bar plot -- mean(|SHAP|) ranking of features.
    shap.plots.bar(explanation, show=False)
    plt.savefig(RESULT_DIR / "sv_interpretability_shap_bar.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 4. Beeswarm plot -- same data as summary, new-API styling.
    shap.plots.beeswarm(explanation, show=False)
    plt.savefig(RESULT_DIR / "sv_interpretability_shap_beeswarm.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 5. Waterfall plot -- explain a single row (E[f(X)] -> f(x) breakdown).
    shap.plots.waterfall(explanation[0], show=False)
    plt.savefig(RESULT_DIR / "sv_interpretability_shap_waterfall_row0.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"SHAP plots saved under: {RESULT_DIR}")

    # -------------------------------------------------------------------------
    # k-SII (pairwise interactions) -- use shapiq's native plots
    # -------------------------------------------------------------------------
    # Only first-order values wrap into shap.Explanation; SHAP's plotting API
    # cannot represent second-order k-SII terms faithfully. Use shapiq plots
    # on the InteractionValues object instead (see also the SHAP-IQ section).

    interaction_explainer = tabpfn_shapiq.get_tabpfn_imputation_explainer(
        model=clf,
        data=X_train,
        index="k-SII",
        imputer="baseline",
        max_order=2,
    )

    KSII_BUDGET = 256

    # Explain one minority-class row (first in X_explain after positive-first ordering).
    _x_ksii = X_explain[0]
    print(f"Computing pairwise Shapley interactions (k-SII, budget={KSII_BUDGET})...")
    with _quiet_tabpfn_client():
        iv_ksii = interaction_explainer.explain(x=_x_ksii, budget=KSII_BUDGET)

    iv_ksii.plot_network(feature_names=list(feature_names))
    plt.savefig(RESULT_DIR / "k_ssi_interpretability_network.png", dpi=150, bbox_inches="tight")
    plt.show()

    iv_ksii.plot_upset(feature_names=list(feature_names))
    plt.savefig(RESULT_DIR / "k_ssi_interpretability_upset.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"k-SII interaction plots saved under: {RESULT_DIR}")

_elapsed = time.perf_counter() - _t0
print(f"[3/4] done in {_elapsed:.1f}s | artifacts under: {RESULT_DIR}")




### SHAP-IQ Part

In [ ]:
"""Shapley values + pairwise Shapley interactions for the VLST TabPFN
classifier using the shapiq library, visualized with shapiq's native
plots.

Two paradigms for "feature removal" are illustrated:

  1. Imputation-based (`get_tabpfn_imputation_explainer`): masked
     features are filled by an imputer (default: baseline -- mean for
     numeric features, mode for categorical, learned from `data`). The
     training set is fixed across coalitions, so the KV-cache fast path
     applies -- construct the model with `fit_mode="fit_with_cache"`.

  2. Remove-and-recontextualize (`get_tabpfn_explainer`): TabPFN is
     re-fit on each coalition's column subset. Does not benefit from
     the KV cache (one predict per fit). Left commented out below --
     prohibitive at VLST's dimensionality.

For classifiers, both explainers default to `class_index=1`, i.e. they
explain P(positive) -- "Stent thrombosis" on VLST.
"""

from __future__ import annotations

import time
import warnings

from sklearn.model_selection import train_test_split

from tabpfn_client import TabPFNClassifier
from tabpfn_extensions.interpretability import shapiq as tabpfn_shapiq

print("=" * 60)
print("[4/4] SHAP-IQ (native shapiq plots)")
print("=" * 60)

_t0 = time.perf_counter()

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE,
)

# Interaction/network/upset plots are most informative on a minority-class
# row when one exists; fall back to the first row otherwise.
_pos = np.where(y_test == 1)[0]
x_explain = X_test[_pos[0] if len(_pos) else 0]

def _make_iq_clf(**extra):
    return TabPFNClassifier(
        balance_probabilities=True,
        thinking_mode=INTERP_THINKING_MODE,
        thinking_effort=INTERP_THINKING_EFFORT,
        thinking_metric=INTERP_THINKING_METRIC,
        random_state=42,
    )

print("Fitting TabPFN for SHAP-IQ (1 thinking fit)...")
try:
    clf = _silence_client_progress(_make_iq_clf(fit_mode="fit_with_cache"))
    with _quiet_tabpfn_client():
        clf.fit(X_train, y_train)
except (TypeError, ValueError, NotImplementedError):
    warnings.warn(
        "shapiq would benefit substantially from the KV cache, but the "
        "current TabPFN install doesn't support fit_mode='fit_with_cache'. "
        "Falling back to the default constructor.",
        UserWarning, stacklevel=2,
    )
    clf = _silence_client_progress(_make_iq_clf())
    with _quiet_tabpfn_client():
        clf.fit(X_train, y_train)

# VLST has ~50 features; 2**d exact enumeration is intractable -- use
# sampled coalitions. Interactions (max_order=2) generally need a bigger
# budget than plain SV. Keep budgets at 256 (do not duplicate SHAP's
# expensive multi-row work -- this section is one-row native plots).
SV_BUDGET = 256
KSII_BUDGET = 256


# -----------------------------------------------------------------------------
# 1. Imputation-based explainer (uses the KV cache)
# -----------------------------------------------------------------------------
imputation_explainer = tabpfn_shapiq.get_tabpfn_imputation_explainer(
    model=clf,
    data=X_train,
    index="SV",     # plain Shapley values
    imputer="baseline",
    max_order=1,
)
print(f"Computing imputation-based Shapley values (1 row, budget={SV_BUDGET})...")
with _quiet_tabpfn_client():
    sv_imp = imputation_explainer.explain(x=x_explain, budget=SV_BUDGET)
sv_imp.plot_force(feature_names=list(feature_names))


# -----------------------------------------------------------------------------
# 2. Pairwise Shapley interactions via the same explainer (k-SII, max_order=2)
# -----------------------------------------------------------------------------
interaction_explainer = tabpfn_shapiq.get_tabpfn_imputation_explainer(
    model=clf,
    data=X_train,
    index="k-SII",  # k-Shapley Interaction Index -- extends SHAP to interactions
    imputer="baseline",
    max_order=2,
)
print(f"Computing pairwise Shapley interactions (k-SII, 1 row, budget={KSII_BUDGET})...")
with _quiet_tabpfn_client():
    iv_interactions = interaction_explainer.explain(x=x_explain, budget=KSII_BUDGET)

# Network plot: features as nodes sized by individual SV; edges colored by
# pairwise interaction strength. Specific to shapiq (not in the shap library).
iv_interactions.plot_network(feature_names=list(feature_names))

# Upset plot of top interactions.
iv_interactions.plot_upset(feature_names=list(feature_names))


# -----------------------------------------------------------------------------
# 3. Remove-and-recontextualize (Rundel) -- much slower, no KV cache.
# -----------------------------------------------------------------------------
# Each coalition triggers a fresh TabPFN fit on a different column subset,
# followed by exactly one predict. On VLST-sized d this is prohibitive.
# Kept as reference -- uncomment only for small feature subsets.

# rundel_explainer = tabpfn_shapiq.get_tabpfn_explainer(
#     model=clf,
#     data=X_train,
#     labels=y_train,
#     index="SV",
#     max_order=1,
# )
# sv_rundel = rundel_explainer.explain(x=x_explain, budget=SV_BUDGET)
# sv_rundel.plot_force(feature_names=list(feature_names))

_elapsed = time.perf_counter() - _t0
print(f"[4/4] done in {_elapsed:.1f}s")


